In [1]:
import pandas as pd
import duckdb
import os
import os, json
from uuid import uuid4
import pandas as pd
import numpy as np
import pickle

In [2]:
works_df = pd.read_parquet("../data/grela_works_metadata.parquet")

In [3]:
works_df.head(5)

,grela_source,grela_id,author,author_viaf,author_wd,title,not_before,not_after,lagt_tlg_epithet,lagt_genre,...,author_gnd,noscemus_place,noscemus_genre,noscemus_discipline,title_short,emlap_noscemus_id,place_publication,place_geonames,title_viaf,date_random
0,lagt,lagt_ggm0001.ggm001,Anonymous,None,None,Anametresis Pontou,1.0,400.0,None,None,...,None,None,None,None,None,NaN,None,None,NaN,298.0
1,lagt,lagt_ogl0001.ogl001,Pinytus,None,None,De Epistola Pinyti ad Dionysium,101.0,200.0,[],[],...,None,None,None,None,None,NaN,None,None,NaN,148.0
2,lagt,lagt_pta0001.pta001,Severian of Gabala,None,None,De fide et lege naturae,400.0,409.0,None,None,...,None,None,None,None,None,NaN,None,None,NaN,406.0
3,lagt,lagt_pta0001.pta002,Severian of Gabala,None,None,De paenitentia et compunctione,400.0,409.0,None,None,...,None,None,None,None,None,NaN,None,None,NaN,400.0
4,lagt,lagt_pta0001.pta003,Severian of Gabala,None,None,In ascensionem domini nostri Iesu Christi et i...,400.0,409.0,None,None,...,None,None,None,None,None,NaN,None,None,NaN,405.0


In [55]:
works_df.columns

Index(['grela_source', 'grela_id', 'author', 'author_viaf', 'author_wd',
       'title', 'not_before', 'not_after', 'lagt_tlg_epithet', 'lagt_genre',
       'lagt_provenience', 'author_gnd', 'noscemus_place', 'noscemus_genre',
       'noscemus_discipline', 'title_short', 'emlap_noscemus_id',
       'place_publication', 'place_geonames', 'title_viaf', 'date_random'],
      dtype='object')

In [56]:
def get_subcorpora_metadata(row):
    subcorpus_meta_dict = {}
    for col in ['lagt_tlg_epithet', 'lagt_genre', 'noscemus_place', 'noscemus_genre', 'noscemus_discipline', 'emlap_noscemus_id']:
        subcorpus_meta_dict[col] = row[col]
    return subcorpus_meta_dict

works_df["subcorpus_specific_metadata"] = works_df.apply(get_subcorpora_metadata, axis=1)

In [57]:
works_df.rename(columns={"lagt_provenience" : "provenience"}, inplace=True)

In [58]:
works_df = works_df[['grela_source', 'grela_id', 'author', 'title', 'not_before', 'not_after', 'date_random', 'provenience', 'place_publication', 'place_geonames', 'author_viaf', 'author_wd', 'author_gnd', 'title_viaf', 'subcorpus_specific_metadata']]

In [2]:
conn = duckdb.connect("/srv/data/grela/grela_v0.6.duckdb")

In [60]:
conn.register("works_df", works_df)

conn.execute("DROP TABLE IF EXISTS works")

conn.execute("""
CREATE TABLE works AS
SELECT
    CAST(grela_source AS VARCHAR) AS grela_source,
    CAST(grela_id AS VARCHAR) AS grela_id,
    CAST(author AS VARCHAR) AS author,
    CAST(title AS VARCHAR) AS title,
    CAST(not_before AS INTEGER) AS not_before,
    CAST(not_after AS INTEGER) AS not_after,
    CAST(date_random AS INTEGER) AS date_random,
    CAST(provenience AS VARCHAR) AS provenience,
    CAST(place_publication AS VARCHAR) AS place_publication,
    CAST(place_geonames AS VARCHAR) AS place_geonames,
    CAST(author_viaf AS VARCHAR) AS author_viaf,
    CAST(author_wd AS VARCHAR) AS author_wd,
    CAST(author_gnd AS VARCHAR) AS author_gnd,
    CAST(title_viaf AS VARCHAR) AS title_viaf,
    CAST(subcorpus_specific_metadata AS JSON) AS subcorpus_specific_metadata
FROM works_df;
""")

In [61]:
print(conn.execute("PRAGMA table_info('works')").fetchdf())

    cid                         name     type  notnull dflt_value     pk
0     0                 grela_source  VARCHAR    False       None  False
1     1                     grela_id  VARCHAR    False       None  False
2     2                       author  VARCHAR    False       None  False
3     3                        title  VARCHAR    False       None  False
4     4                   not_before  INTEGER    False       None  False
5     5                    not_after  INTEGER    False       None  False
6     6                  date_random  INTEGER    False       None  False
7     7                  provenience  VARCHAR    False       None  False
8     8            place_publication  VARCHAR    False       None  False
9     9               place_geonames  VARCHAR    False       None  False
10   10                  author_viaf  VARCHAR    False       None  False
11   11                    author_wd  VARCHAR    False       None  False
12   12                   author_gnd  VARCHAR    Fa

In [20]:
# Drop old tables if needed (optional but clean)
conn.execute("DROP TABLE IF EXISTS sentences;")
conn.execute("DROP TABLE IF EXISTS tokens;")

# Create with correct types
conn.execute("""
    CREATE TABLE sentences (
        sentence_id VARCHAR,
        grela_id VARCHAR,
        position INTEGER,
        sent_text VARCHAR
    );
""")

conn.execute("""
    CREATE TABLE tokens (
        sentence_id VARCHAR,
        grela_id VARCHAR,
        token_text VARCHAR,
        lemma VARCHAR,
        pos VARCHAR,
        ref JSON,             -- real JSON type
        char_start INTEGER,
        char_end INTEGER,
        token_id INTEGER,
    );
""")

In [3]:
def process_single_file(filepath, grela_prefix, conn, sentences_table="sentences", tokens_table="tokens", replace=False):
    """
    Loads a single JSON file and converts it into
    sentences_df and tokens_df.
    If grela_id already exists in DB and replace=False → SKIP.
    """

    fn = os.path.basename(filepath)
    grela_id = f"{grela_prefix}_{fn.replace('.json', '')}"

    # --- NEW: check DB before parsing JSON ---
    exists = conn.execute(
        f"SELECT COUNT(*) FROM {sentences_table} WHERE grela_id = ?",
        [grela_id]
    ).fetchone()[0]

    if exists > 0 and not replace:
        print(f"Skipping {fn}: grela_id={grela_id} already in DB.")
        return None, None, None
    # -------------------------------------------

    # Load JSON file
    with open(filepath, "r", encoding="utf-8") as f:
        sents_data = json.load(f)

    sentences = []
    tokens = []

    for sent in sents_data:
        sent_id = int(sent["sent_id"])
        sentence_id = f"{grela_id}_{sent_id}"

        # Build sentence row
        sentences.append({
            "sentence_id": sentence_id,
            "grela_id": grela_id,
            "position": sent_id,
            "sent_text": sent["sent_text"]
        })

        # Tokens: tolerate both "token_data" and "tokens_data"
        token_list = sent.get("tokens_data", sent.get("token_data", []))

        for tok in token_list:
            tokens.append({
                "sentence_id": sentence_id,
                "grela_id": grela_id,
                "token_text": tok["token_text"],
                "lemma": tok["lemma"],
                "pos": tok["pos"],
                "ref": json.dumps(tok["ref"]),      # Convert dict → JSON string
                "char_start": tok["char_start"],
                "char_end": tok["char_end"],
                "token_id" : 0
            })

    return grela_id, pd.DataFrame(sentences), pd.DataFrame(tokens)

def store_processed_data(conn, grela_id,
                         sentences_df, tokens_df,
                         sentences_table="sentences",
                         tokens_table="tokens",
                         replace=False):
    """
    Stores sentences and tokens into DuckDB.
    If replace=True, removes old rows for this grela_id first.
    """

    if replace:
        conn.execute(f"DELETE FROM {sentences_table} WHERE grela_id = ?", [grela_id])
        conn.execute(f"DELETE FROM {tokens_table} WHERE grela_id = ?", [grela_id])

    # Register temp tables
    conn.register("temp_sentences", sentences_df)
    conn.register("temp_tokens", tokens_df)

    # Insert
    conn.execute(f"INSERT INTO {sentences_table} SELECT * FROM temp_sentences")
    conn.execute(f"INSERT INTO {tokens_table} SELECT * FROM temp_tokens")

    # Cleanup
    conn.unregister("temp_sentences")
    conn.unregister("temp_tokens")


def process_and_store_in_duckdb(
        dir_path, grela_prefix, conn,
        sentences_table="sentences",
        tokens_table="tokens",
        replace=False):

    for fn in os.listdir(dir_path):
        if not fn.endswith(".json"):
            continue

        filepath = os.path.join(dir_path, fn)

        try:
            grela_id, sentences_df, tokens_df = process_single_file(
                filepath, grela_prefix, conn,
                sentences_table=sentences_table,
                tokens_table=tokens_table,
                replace=replace
            )

            # If processing was skipped
            if grela_id is None:
                continue

            store_processed_data(
                conn, grela_id,
                sentences_df, tokens_df,
                sentences_table=sentences_table,
                tokens_table=tokens_table,
                replace=replace
            )

            print(f"Processed {fn}")

        except Exception as e:
            print(f"Failed processing {fn}: {e}")

In [4]:
lagt_sents_data_dir = "/home/jupyter-vojta/notebooks/LAGT/data/large_files/sents_data_jsons_dicts"
emlap_sents_data_dir =  "/home/jupyter-vojta/notebooks/EMLAP_ETL/data/sents_data_jsons_dicts" # "/srv/data/tome/tome-corpus/sents_data_id_jsons_v3-0/"
noscemus_sents_data_dir = "../../noscemus_ETF/data/sents_data_jsons_dicts/"
cc_sents_data_dir = "/srv/data/corpus-corporum/sents_jsons_dicts/"
vulgate_data_dir = "../data/vulgate_sentences_dicts"

In [13]:
process_and_store_in_duckdb(lagt_sents_data_dir, "lagt", conn, replace=False)

Skipping tlg0007.tlg070.json: grela_id=lagt_tlg0007.tlg070 already in DB.
Skipping tlg2586.tlg002.json: grela_id=lagt_tlg2586.tlg002 already in DB.
Skipping tlg1271.tlg003.json: grela_id=lagt_tlg1271.tlg003 already in DB.
Skipping tlg0533.tlg020.json: grela_id=lagt_tlg0533.tlg020 already in DB.
Skipping tlg0086.tlg054.json: grela_id=lagt_tlg0086.tlg054 already in DB.
Skipping tlg0030.tlg003.json: grela_id=lagt_tlg0030.tlg003 already in DB.
Skipping pta9999.pta003.json: grela_id=lagt_pta9999.pta003 already in DB.
Skipping tlg2042.tlg086.json: grela_id=lagt_tlg2042.tlg086 already in DB.
Skipping tlg1607.tlg002.json: grela_id=lagt_tlg1607.tlg002 already in DB.
Skipping tlg0032.tlg005.json: grela_id=lagt_tlg0032.tlg005 already in DB.
Skipping pta0001.pta001.json: grela_id=lagt_pta0001.pta001 already in DB.
Skipping pta9999.pta032.json: grela_id=lagt_pta9999.pta032 already in DB.
Skipping tlg0284.tlg044.json: grela_id=lagt_tlg0284.tlg044 already in DB.
Skipping tlg2018.tlg005.json: grela_id

In [24]:
process_and_store_in_duckdb(cc_sents_data_dir, "cc", conn, replace=False)

Processed 13339.json
Processed 21349.json
Processed 9388.json
Processed 14847.json
Processed 7119.json
Processed 21369.json
Processed 10788.json
Processed 10787.json
Processed 19946.json
Processed 10394.json
Processed 10174.json
Processed 11466.json
Processed 10062.json
Processed 8688.json
Processed 102.json
Processed 7669.json
Processed 12140.json
Processed 7663.json
Processed 42.json
Processed 19907.json
Processed 12213.json
Processed 10721.json
Processed 9034.json
Processed 11397.json
Processed 11713.json
Processed 14969.json
Processed 19695.json
Processed 13048.json
Processed 12700.json
Processed 10924.json
Processed 9224.json
Processed 21389.json
Processed 12627.json
Processed 21639.json
Processed 9960.json
Processed 9632.json
Processed 20062.json
Processed 9078.json
Processed 13061.json
Processed 19873.json
Processed 8189.json
Processed 8143.json
Processed 9633.json
Processed 9439.json
Processed 8240.json
Processed 8572.json
Processed 9788.json
Processed 9104.json
Processed 7863.

In [25]:
process_and_store_in_duckdb(vulgate_data_dir, "vulgate", conn, replace=False)

Processed tlg0527.tlg020.obi-lat.json
Processed tlg0527.tlg013.obi-lat.json
Processed tlg0527.tlg019.obi-lat.json
Processed tlg0527.tlg016.obi-lat.json
Processed tlg0527.tlg021.obi-lat.json
Processed tlg0527.tlg045.obi-lat.json
Processed tlg0527.tlg053.obi-lat.json
Processed tlg0527.tlg003.obi-lat.json
Processed tlg0527.tlg048.obi-lat.json
Processed tlg0031.tlg011.obi-lat.json
Processed tlg0527.tlg010.obi-lat.json
Processed tlg0527.tlg002.obi-lat.json
Processed tlg0527.tlg056.obi-lat.json
Processed tlg0527.tlg005.obi-lat.json
Processed tlg0031.tlg019.obi-lat.json
Processed tlg0031.tlg017.obi-lat.json
Processed tlg0031.tlg005.obi-lat.json
Processed tlg0031.tlg012.obi-lat.json
Processed tlg0527.tlg006.obi-lat.json
Processed tlg0031.tlg023.obi-lat.json
Processed tlg0031.tlg020.obi-lat.json
Processed tlg0527.tlg041.obi-lat.json
Processed tlg0031.tlg027.obi-lat.json
Processed tlg0527.tlg039.obi-lat.json
Processed tlg0527.tlg030.obi-lat.json
Processed tlg0527.tlg015.obi-lat.json
Processed tl

In [28]:
process_and_store_in_duckdb(noscemus_sents_data_dir, "noscemus", conn, replace=False)

Processed 604894.json
Processed 795551.json
Processed 830008.json
Processed 694621.json
Processed 732897.json
Processed 731628.json
Processed 668508.json
Processed 906967.json
Processed 705020.json
Processed 831540.json
Processed 658379.json
Processed 902261.json
Processed 807330.json
Processed 845314.json
Processed 605913.json
Processed 747384.json
Processed 888131.json
Processed 739097.json
Processed 599729.json
Processed 913064.json
Processed 695816.json
Processed 668505.json
Processed 742067.json
Processed 704812.json
Processed 907179.json
Failed processing 906966.json: Invalid Input Error: Need a DataFrame with at least one column
Processed 704336.json
Processed 725081.json
Processed 710274.json
Processed 735139.json
Processed 743611.json
Processed 816417.json
Processed 725080.json
Processed 623155.json
Processed 668500.json
Processed 1370560.json
Processed 904631.json
Processed 897258.json
Processed 888133.json
Processed 658513.json
Processed 900767.json
Processed 604891.json
Pro

In [5]:
process_and_store_in_duckdb(emlap_sents_data_dir, "emlap", conn, replace=True)

Processed 100044.json
Processed 100034.json
Processed 100014.json
Processed 100094.json
Processed 100060.json
Processed 100090.json
Processed 100010.json
Processed 100078.json
Processed 100068.json
Processed 100079.json
Processed 100043.json
Processed 100072.json
Processed 100041.json
Processed 100012.json
Processed 100082.json
Processed 100025.json
Processed 100019.json
Processed 100085.json
Processed 100093.json
Processed 100095.json
Processed 100089.json
Processed 100047.json
Processed 100086.json
Processed 100013.json
Processed 100028.json
Processed 100073.json
Processed 100070.json
Processed 100048.json
Processed 100081.json
Processed 100088.json
Processed 100038.json
Processed 100035.json
Processed 100042.json
Processed 100071.json
Processed 100053.json
Processed 100083.json
Processed 100003.json
Processed 100059.json
Processed 100002.json
Processed 100049.json
Processed 100032.json
Processed 100052.json
Processed 100066.json
Processed 100045.json
Processed 100096.json
Processed 

In [6]:
# simple test with insterted new tokens
query = """
SELECT *
FROM sentences
WHERE grela_id LIKE 'lagt_tlg0086%';
"""
result_df = conn.execute(query).fetchdf()
result_df[:10]

,sentence_id,grela_id,position,sent_text
0,lagt_tlg0086.tlg054_0,lagt_tlg0086.tlg054,0,ἅπαν τὸ κινούμενον ἀνάγκη ὑπό τινος κινεῖσθαι.
1,lagt_tlg0086.tlg054_1,lagt_tlg0086.tlg054,1,εἰ μὲν οὖν ἐν αὐτῷ μὴ ἔχει τὴν ἀρχὴν τῆς κινήσ...
2,lagt_tlg0086.tlg054_2,lagt_tlg0086.tlg054,2,"εἰ δ ἐν αὐτῷ, εἰλήφθω ἐφ οὗ τὸ ΑΒ, ὃ κινεῖται ..."
3,lagt_tlg0086.tlg054_3,lagt_tlg0086.tlg054,3,πρῶτον μὲν οὖν τὸ ὑπολαμβάνειν τὸ ΑΒ ὑφ αὐτοῦ ...
4,lagt_tlg0086.tlg054_4,lagt_tlg0086.tlg054,4,ἔτι τὸ ὑφ αὐτοῦ κινούμενον οὐδέποτε παύσεται κ...
5,lagt_tlg0086.tlg054_5,lagt_tlg0086.tlg054,5,"ἀνάγκη τοίνυν, εἴ τι παύεται κινούμενον τῷ ἕτε..."
6,lagt_tlg0086.tlg054_6,lagt_tlg0086.tlg054,6,τούτου δὲ φανεροῦ γενομένου ἀνάγκη πᾶν τὸ κινο...
7,lagt_tlg0086.tlg054_7,lagt_tlg0086.tlg054,7,"ἐπεὶ γὰρ εἴληπται τὸ ΑΒ κινούμενον, διαιρετὸν ..."
8,lagt_tlg0086.tlg054_8,lagt_tlg0086.tlg054,8,πᾶν γὰρ τὸ κινούμενον διαιρετὸν ἦν.
9,lagt_tlg0086.tlg054_9,lagt_tlg0086.tlg054,9,διῃρήσθω τοίνυν ᾗ τὸ Γ.


In [7]:
# Step 1: Add the `token_id` column if it does not already exist
try:
    conn.execute("ALTER TABLE tokens ADD COLUMN token_id BIGINT;")
except Exception as e:
    if "Column with name token_id already exists" not in str(e):
        raise

conn.execute("UPDATE tokens SET token_id = rowid;")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [8]:
conn.execute("SHOW TABLES").fetchdf()

,name
0,sentences
1,tokens
2,works


In [9]:
# Query to list all tables in the database
tables_query = "SHOW TABLES;"
tables = conn.execute(tables_query).fetchall()

In [10]:
for table in tables:
    print(conn.execute(f"SELECT COUNT(*) as row_count FROM {table[0]};").fetchdf())

   row_count
0   21539273
   row_count
0  356325725
   row_count
0      11127


In [11]:
conn.execute("""-- add the column once
ALTER TABLE works ADD COLUMN token_count BIGINT DEFAULT 0;

-- (re-)populate in a single statement
UPDATE works AS w
SET    token_count = sub.cnt
FROM (
        SELECT grela_id, COUNT(*) AS cnt
        FROM tokens
        GROUP BY grela_id
     ) AS sub
WHERE w.grela_id = sub.grela_id;
""")

In [12]:
conn.execute("""
-- Add the column once (if not already created)
ALTER TABLE works ADD COLUMN sentence_count BIGINT DEFAULT 0;

-- (Re-)populate in a single statement
UPDATE works AS w
SET    sentence_count = sub.cnt
FROM (
        SELECT grela_id, COUNT(*) AS cnt
        FROM sentences
        GROUP BY grela_id
     ) AS sub
WHERE w.grela_id = sub.grela_id;
""")

In [13]:
# 1️⃣ Compute aggregated stats per grela_source
df = conn.execute("""
    SELECT
        grela_source,
        COUNT(*) AS works_N,
        SUM(sentence_count) AS sentences_N,
        SUM(token_count) AS tokens_N
    FROM works
    GROUP BY grela_source;
""").fetchdf()

# 2️⃣ Convert numeric aggregates to int + comma formatting
numeric_cols = ["works_N", "sentences_N", "tokens_N"]
for col in numeric_cols:
    df[col] = df[col].astype("int64")
    df[col] = df[col].map("{:,}".format)

# 3️⃣ Your required fixed ordering
order = ["lagt", "cc", "noscemus", "emlap", "vulgate"]

df["order"] = df["grela_source"].apply(lambda x: order.index(x))
df = df.sort_values("order").drop(columns=["order"])

# 4️⃣ Produce Markdown
md = df.to_markdown(index=False)
print(md)

| grela_source   | works_N   | sentences_N   | tokens_N    |
|:---------------|:----------|:--------------|:------------|
| lagt           | 2,160     | 2,095,265     | 38,223,149  |
| cc             | 7,819     | 14,229,691    | 254,770,887 |
| noscemus       | 975       | 4,637,231     | 54,542,448  |
| emlap          | 100       | 444,211       | 6,477,016   |
| vulgate        | 73        | 35,254        | 603,091     |


In [14]:
# Query to get table and column information
query = """
    SELECT
        table_name,
        column_name,
        data_type,
        is_nullable,
        column_default
    FROM information_schema.columns
    ORDER BY table_name, ordinal_position
"""

# Execute the query and fetch the schema information as a DataFrame
df = conn.execute(query).fetchdf()

# Group the schema details by table
tables = df.groupby("table_name")

# Markdown generation
markdown = "# Database Schema Documentation\n\n"
for table_name, group in tables:
    markdown += f"## Table: `{table_name}`\n\n"
    markdown += "| Column Name     | Data Type    | Is Nullable | Default Value |\n"
    markdown += "|-----------------|-------------|-------------|---------------|\n"

    for _, row in group.iterrows():
        markdown += (
            f"| {row['column_name']} | {row['data_type']} | "
            f"{row['is_nullable']} | {row['column_default'] or 'N/A'} |\n"
        )

    markdown += "\n"  # Add a space between tables


In [15]:
print(markdown)

# Database Schema Documentation

## Table: `sentences`

| Column Name     | Data Type    | Is Nullable | Default Value |
|-----------------|-------------|-------------|---------------|
| sentence_id | VARCHAR | YES | N/A |
| grela_id | VARCHAR | YES | N/A |
| position | INTEGER | YES | N/A |
| sent_text | VARCHAR | YES | N/A |

## Table: `tokens`

| Column Name     | Data Type    | Is Nullable | Default Value |
|-----------------|-------------|-------------|---------------|
| sentence_id | VARCHAR | YES | N/A |
| grela_id | VARCHAR | YES | N/A |
| token_text | VARCHAR | YES | N/A |
| lemma | VARCHAR | YES | N/A |
| pos | VARCHAR | YES | N/A |
| ref | JSON | YES | N/A |
| char_start | INTEGER | YES | N/A |
| char_end | INTEGER | YES | N/A |
| token_id | BIGINT | YES | N/A |

## Table: `works`

| Column Name     | Data Type    | Is Nullable | Default Value |
|-----------------|-------------|-------------|---------------|
| grela_source | VARCHAR | YES | N/A |
| grela_id | VARCHAR | YES |

In [16]:
conn.close()